# EDA & Fraud Detection Notebook

This notebook performs **Exploratory Data Analysis (EDA)** and builds fraud detection models on the `bank_transactions_data_2.csv` dataset.

We:
- Explore transaction patterns and customer behaviour
- Create a synthetic `FraudFlag` using a **score-based rule** (flag when ≥ 2 of 6 indicators fire)
- Preprocess features
- Train multiple classification models
- Compare their performance

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier
from sklearn.tree import plot_tree, DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
print('Libraries loaded successfully.')

## 1. Load Data

In [ ]:
df = pd.read_csv('bank_transactions_data_2.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('Shape:', df.shape, '\n')
print('Info:')
df.info()

print('\nDescribe (numeric):')
display(df.describe())

print('Missing values per column:')
print(df.isnull().sum())

### 2.1 Histograms – Numeric Columns

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns

df[numeric_cols].hist(figsize=(14, 10), bins=30)
plt.suptitle('Histograms of Numeric Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 2.2 Boxplots – Outlier Detection

In [ ]:
plt.figure(figsize=(14, 8))
df[numeric_cols].boxplot()
plt.title('Boxplots of Numeric Features')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

### 2.3 Correlation Heatmap

In [ ]:
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap='coolwarm', interpolation='nearest')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticklabels(corr.columns)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

### 2.4 Categorical Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df['TransactionType'].value_counts().plot(kind='bar', ax=axes[0],
    color='steelblue', edgecolor='black')
axes[0].set_title('Transaction Type')
axes[0].tick_params(axis='x', rotation=0)

df['Channel'].value_counts().plot(kind='bar', ax=axes[1],
    color='coral', edgecolor='black')
axes[1].set_title('Channel')
axes[1].tick_params(axis='x', rotation=0)

df['CustomerOccupation'].value_counts().plot(kind='bar', ax=axes[2],
    color='mediumseagreen', edgecolor='black')
axes[2].set_title('Customer Occupation')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Categorical Feature Distributions', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Feature Engineering – Synthetic Fraud Flag

Each transaction is scored against **6 risk indicators**. A transaction is flagged as fraud when it triggers **2 or more** indicators.

| # | Indicator | Detail |
|---|-----------|--------|
| 1 | Credit → immediate Debit | Time diff ≤ 300 s |
| 2 | Non-US location | Location not in known US-city list |
| 3 | Suspicious IP | First octet not in US IP-block list |
| 4 | Age 20–35 | `CustomerAge` between 20 and 35 |
| 5 | High login attempts | `LoginAttempts` > 3 |
| 6 | Short transaction duration | `TransactionDuration` < 30 s |

> **Score ≥ 2 → FraudFlag = 1** (balances sensitivity vs. specificity)

In [ ]:
# ── Parse datetimes ──────────────────────────────────────────────────────
df['TransactionDate']         = pd.to_datetime(df['TransactionDate'])
df['PreviousTransactionDate'] = pd.to_datetime(df['PreviousTransactionDate'])
df = df.sort_values(['AccountID', 'TransactionDate'])

# Time difference in seconds
df['time_diff_sec'] = (
    df['TransactionDate'] - df['PreviousTransactionDate']
).dt.total_seconds()

# ── Indicator 1: Credit with immediate follow-up (≤300 s gap) ────────────
ind1 = ((df['TransactionType'] == 'Credit') &
         df['time_diff_sec'].between(0, 300, inclusive='both')).astype(int)

# ── Indicator 2: Non-US location ─────────────────────────────────────────
us_cities = [
    'San Diego','Houston','Mesa','Raleigh','Atlanta','Oklahoma City',
    'Seattle','Indianapolis','Detroit','Nashville','Albuquerque','Memphis',
    'Miami','Milwaukee','Las Vegas','New York','San Francisco','Chicago',
    'Denver','Dallas','San Jose','San Antonio','Philadelphia','Boston',
    'Jacksonville','Sacramento','Washington','Portland','Colorado Springs',
    'El Paso','Virginia Beach','Baltimore','Phoenix','Fort Worth',
    'Kansas City','Fresno','Charlotte','Tucson','Omaha','Columbus','Louisville'
]
ind2 = (~df['Location'].isin(us_cities)).astype(int)

# ── Indicator 3: Suspicious IP block ─────────────────────────────────────
us_blocks = {
    3,4,6,7,11,12,13,15,16,17,18,20,23,24,26,28,32,33,34,35,38,
    40,44,47,52,54,56,63,64,65,66,67,68,69,70,71,72,73,74,75,76,96,97,98,99
}
def is_us_ip(ip):
    try: return int(str(ip).split('.')[0]) in us_blocks
    except: return False

df['is_us_ip'] = df['IP Address'].apply(is_us_ip)
ind3 = (~df['is_us_ip']).astype(int)

# ── Indicators 4-6 ───────────────────────────────────────────────────────
ind4 = df['CustomerAge'].between(20, 35, inclusive='both').astype(int)
ind5 = (df['LoginAttempts'] > 3).astype(int)
ind6 = (df['TransactionDuration'] < 30).astype(int)

# ── Score & threshold ─────────────────────────────────────────────────────
df['fraud_score'] = ind1 + ind2 + ind3 + ind4 + ind5 + ind6
FRAUD_THRESHOLD   = 2          # flag when 2+ indicators fire
df['FraudFlag']   = (df['fraud_score'] >= FRAUD_THRESHOLD).astype(int)

print('Fraud score distribution:')
print(df['fraud_score'].value_counts().sort_index())
print(f'\nFraudFlag counts (threshold = {FRAUD_THRESHOLD}):')
print(df['FraudFlag'].value_counts())
print(f'\nFraud rate: {df["FraudFlag"].mean():.2%}')

In [ ]:
preview_cols = [
    'TransactionID','AccountID','TransactionType','Location',
    'IP Address','CustomerAge','LoginAttempts','TransactionDuration',
    'fraud_score','FraudFlag'
]
df[preview_cols].head(20)

### 3.1 Fraud Flag Distribution

In [ ]:
# ── Build dynamic labels and colours (works for 1 or 2 unique values) ──
label_map  = {0: 'Legitimate (0)', 1: 'Fraud (1)'}
colour_map = {0: 'steelblue',      1: 'crimson'}

counts     = df['FraudFlag'].value_counts().sort_index()
bar_labels = [label_map[k]  for k in counts.index]
bar_colors = [colour_map[k] for k in counts.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Bar chart ─────────────────────────────────────────────────────────────
bars = axes[0].bar(bar_labels, counts.values,
                   color=bar_colors, edgecolor='black')
axes[0].set_title('Fraud vs Legitimate Transactions')
axes[0].set_ylabel('Count')
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + max(counts.values) * 0.01,
                 str(v), ha='center', fontweight='bold')

# ── Pie chart – labels always match counts length ─────────────────────────
pie_labels = [label_map[k] for k in counts.index]
pie_colors = [colour_map[k] for k in counts.index]

axes[1].pie(counts.values,
            labels=pie_labels,
            colors=pie_colors,
            autopct='%1.2f%%',
            startangle=90)
axes[1].set_title('Fraud Proportion')

plt.suptitle('FraudFlag Distribution', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df.groupby('TransactionType')['FraudFlag'].mean().plot(
    kind='bar', ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('Fraud Rate by Transaction Type')
axes[0].set_ylabel('Fraud Rate')
axes[0].tick_params(axis='x', rotation=0)

df.groupby('Channel')['FraudFlag'].mean().plot(
    kind='bar', ax=axes[1], color='mediumseagreen', edgecolor='black')
axes[1].set_title('Fraud Rate by Channel')
axes[1].set_ylabel('Fraud Rate')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
drop_cols = [
    'TransactionID', 'IP Address', 'DeviceID',
    'PreviousTransactionDate', 'TransactionDate',
    'is_us_ip', 'fraud_score'
]
df_model = df.drop(columns=drop_cols)

X = df_model.drop(columns=['FraudFlag'])
y = df_model['FraudFlag']

cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(include=[np.number]).columns
print('Categorical columns:', list(cat_cols))
print('Numeric columns    :', list(num_cols))

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print('\nEncoded shape:', X_encoded.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.3, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} rows | Test: {len(X_test)} rows')
print(f'Train fraud rate: {y_train.mean():.2%} | Test fraud rate: {y_test.mean():.2%}')

## 5. Model Training

Seven models are trained and compared:
- Logistic Regression
- K-Nearest Neighbours (KNN)
- Support Vector Classifier (SVC)
- Random Forest
- Decision Tree
- Bagging (Decision Tree base)
- Voting Classifier (LR + RF + SVC, soft voting)

In [ ]:
import sklearn
_sk_ver = tuple(int(x) for x in sklearn.__version__.split('.')[:2])

models = {}

models['LogisticRegression'] = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])

models['KNN'] = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', KNeighborsClassifier(n_neighbors=5))
])

models['SVC'] = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(kernel='rbf', probability=True))
])

models['RandomForest'] = RandomForestClassifier(
    n_estimators=200, random_state=42, class_weight='balanced'
)

models['DecisionTree'] = DecisionTreeClassifier(
    max_depth=None, random_state=42, class_weight='balanced'
)

base_dt = DecisionTreeClassifier(random_state=42)
if _sk_ver >= (1, 2):
    models['Bagging'] = BaggingClassifier(
        estimator=base_dt, n_estimators=50, random_state=42)
else:
    models['Bagging'] = BaggingClassifier(
        base_estimator=base_dt, n_estimators=50, random_state=42)

models['Voting'] = VotingClassifier(
    estimators=[
        ('lr',  models['LogisticRegression']),
        ('rf',  models['RandomForest']),
        ('svc', models['SVC'])
    ],
    voting='soft'
)

print('Models defined:', list(models.keys()))
print('scikit-learn version:', sklearn.__version__)

In [ ]:
results = {}

for name, model in models.items():
    print(f'\n{"="*55}')
    print(f'  Training: {name}')
    print(f'{"="*55}')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f'Accuracy: {acc:.4f}')
    print(classification_report(y_test, y_pred, digits=4,
                                 target_names=['Legitimate', 'Fraud']))

## 6. Model Comparison

In [ ]:
sorted_results = dict(sorted(results.items(), key=lambda x: x[1], reverse=True))

plt.figure(figsize=(12, 6))
bars = plt.bar(sorted_results.keys(), sorted_results.values(),
               color='steelblue', edgecolor='black')
plt.ylim(min(sorted_results.values()) - 0.05, 1.005)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison')
plt.xticks(rotation=20, ha='right')
for bar, acc in zip(bars, sorted_results.values()):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.001,
             f'{acc:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

print('\nFinal accuracy summary:')
for name, acc in sorted_results.items():
    print(f'  {name:<22} {acc:.4f}')

## 7. Random Forest – Example Tree

One tree from the Random Forest ensemble (max depth = 3 for readability).

In [ ]:
rf = models['RandomForest']

plt.figure(figsize=(20, 10))
plot_tree(
    rf.estimators_[0], max_depth=3,
    feature_names=X_encoded.columns,
    class_names=['Legitimate', 'Fraud'],
    filled=True, fontsize=8
)
plt.title('Random Forest – Example Tree (max depth = 3)', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(
    rf.feature_importances_, index=X_encoded.columns
).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
importances.head(15).plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Top 15 Feature Importances – Random Forest')
plt.ylabel('Importance')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

print(importances.head(10).to_string())

## 9. Conclusion

In this notebook we:

- Performed **EDA** on `bank_transactions_data_2.csv`
- Engineered a synthetic **`FraudFlag`** using a score-based approach (≥ 2 of 6 indicators)
- One-hot encoded categorical features and built **7 classification models**
- Compared models using accuracy and full classification reports
- Visualised feature importances from the Random Forest

### Potential Extensions

| Area | Improvement |
|------|-------------|
| Labels | Replace synthetic rules with real fraud labels |
| Features | IP geolocation, velocity features, graph-based account linking |
| Evaluation | ROC-AUC, Precision-Recall curves, cost-sensitive metrics |
| Modelling | XGBoost/LightGBM, SMOTE for class imbalance, hyperparameter tuning |
| Deployment | Model serialisation (joblib), REST API endpoint |